# SPACESHIP TITANIC CHALLENGE  

---


---

## Table of Contents:  
1. [Introduction](#1.-introduction)  
2. [Setup](#2.-setup)   
3. [Missing Values](#3.-missing-values)  
4. [Exploring the Data](#4.-exploring-the-data)  
5. [Prediction](#5.-prediction)  
6. [Conclusion & Next Steps](#6.-conclusion-&-next-steps)


## 1. Introduction
---
This notebook explores the Spaceship Titanic dataset from Kaggle through manual 
exploratory data analysis (EDA) using only pandas, and a hand-built scorecard model, 
achieving 78.7% accuracy on Kaggle without the use of Machine Learning libraries.

The Spaceship Titanic challenge aims to determine the fate of each pasenger who travels on board the ship.
Given passenger data (age, cabin, spending habits, home planet...), we try to predict which passengers are likely to disapper in a space anomaly.

---

Dataset description provided:
| Column | Description |
|---|---|
| PassengerId | Unique ID in format GroupNumber_PersonNumber |
| HomePlanet | Planet the passenger departed from |
| CryoSleep | Whether the passenger was traveling in a cryo pod |
| Cabin | Cabin number in format Deck/Cabin/Side |
| Destination | Planet the passenger is travelling to |
| Age | Age of the passenger |
| VIP | Whether the passenger paid for VIP service |
| RoomService | Amount spent at the room service facility |
| FoodCourt | Amount spent at the food court |
| ShoppingMall | Amount spent at the shopping mall |
| Spa | Amount spent at the spa |
| VRDeck | Amount spent at the VR deck |
| Name | Passenger name |
| Transported | Whether the passenger was transported by the anomaly|

## 2. Setup
---

In [ ]:
import pandas as pd
data = pd.read_csv('train.csv', index_col='PassengerId')

## 3. Missing Values
---
Let's get some info about the dataset first:

In [ ]:
data.info()
data.head()

We see that all columns have a few missing values, first thing we do will fill them up. We'll use the median for numerical data, and the mode for strings. Let's make a quick function to do it easily.

In [ ]:
def fill_missing(data):
    """ Filling the missing values for each column, using mode and median for the time being. 
    Important note: filling with mode will create groups of ~200 people with the same data.
    Will find a better way to handle missing values after learning more."""
    for column in data.columns:
        if data[column].dtype in ('str', 'object', 'bool'):
            data[column] = data[column].fillna(data[column].mode()[0])
        else:
            data[column] = data[column].fillna(data[column].median())
    return data


fill_missing(data)
data.isnull().sum()

## 4. Exploring the Data
---
Now let's see what we can get from the data.

In [ ]:
print(f'Passenger count: {len(data.index)}')
print((data['HomePlanet'].value_counts(normalize=True) * 100).round())
print((data.groupby('HomePlanet').Transported.mean() * 100).round())


Earth passengers make up 55% of the ship, but have the lowest Transporation rate(43%).  
On the other hand, 66% of Europa passengers were affected by the singularity.  

In [ ]:
print(data['VIP'].value_counts(normalize=True) * 100)
print(data.groupby('VIP').Transported.mean() * 100)

Only 2% of the passengers had VIP status, for a Transporation rate of 38%. It's better than non-VIP passengers, but since it's such a small number of people,the impact of this column isn't significant.

In [ ]:
data.groupby(['HomePlanet', 'CryoSleep'])['Transported'].agg(['mean', 'count'])

99% Transporation rate for Europa passengers in cryosleep, and 91% for Mars passengers in Cryosleep. The pattern is very strong here. Earth is at 67% so will need breaking it down.
The pattern appears clear so far: Europa and Mars have higher risks, and CryoSleep greatly increases the risks across all planets.

In [ ]:
data.groupby(['HomePlanet', 'Destination'])['Transported'].agg(['mean', 'count'])

Breaking donwn Destination and Transported rate by home planet.   
Earth > TRAPPIST and Mars > PSO seem to diverge from the average. 
- Mars > PSO is weak, only 49 people, not worth looking into.
- Earth > TRAPPIST is 3300 people, the difference could perhaps mean something, though 60/40 isn't strong.

In [ ]:
data['AgeGroup'] = pd.cut(data['Age'], bins=[0,5,10,16,25,40,60,100]) 
pd.crosstab([data['HomePlanet'], data['CryoSleep']], data['AgeGroup']).style.format("{:,}")

In [ ]:
print('Transporation rate for each Age Group split by Cryo Sleep')
pd.crosstab(data['CryoSleep'], data['AgeGroup'], values=data['Transported'], aggfunc='mean').style.format("{:.1%}")

This confirms that cryosleep is a very strong indicator for Transportation.
- People from Earth were more likely to have babies in Cryo Sleep. Overall, Earthians were mostly in the 16-25 year old group.
- People from Europa were less likely to have babies and young children in Cryo Sleep, but more likely to have teenagers there. Overall they were mostly in the 25-40 year old group.  
More useful information:
- Young children below 10 are barely affected by Cryo Sleep, the Transporation rate is roughly the same.

In [ ]:
data.groupby('HomePlanet')[['RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']].mean().style.format("{:,.2f}")


- Earthians don't stand out in any luxury activity.
- Europans are very big spenders on the Food Court, Spa, and VR Deck.
- Marsians are big spenders on RoomService & Shopping Mall.

In [ ]:
data.groupby('Transported')[['RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']].mean().style.format("{:,.2f}")


- Heavy Spenders on Room Service, Spa, and VR Deck tend to be a lot safer.
- Food Court & Shopping Mall were in bigger danger, especially the Food Court.  

This can be useful to determine which Europans & Marsians are likely to be transported, depending on which servives they use. Meaningless for Earthians.  

In [ ]:
data['LastName'] = data['Name'].map(lambda n: n.rsplit()[-1])
data['FamilySize'] = data['LastName'].map(data['LastName'].value_counts())
data.groupby('FamilySize')['Transported'].mean()

Trying to detect families, grouping people by last name. No pattern detected, everything is around 50%
Let's try a different approach. Passenger ID should tell us if the passengers traveled in a group. We can also group people by cabin number.

In [ ]:
data['PassengerGroup'] = data.index.map(lambda g: g.split('_')[0]) # Creates a column for each passenger with their group number
data['GroupSize'] = data['PassengerGroup'].map(data['PassengerGroup'].value_counts()) # Creates a column with the group size for each passenger
data[data['GroupSize'] > 1].groupby('PassengerGroup')['Transported'].mean().value_counts() 
# Calculates how many times groups of people were transported together, for groups of more than 1 person.


The interesting Data is if groups of people were transported together, so the 1.0 and the 0.00 values on this table.
In total 615 groups were transported or remained safe together, out of a total of 1400 groups. No pattern detected here.

In [ ]:
print(data.groupby('GroupSize')['Transported'].agg(['mean', 'count']))

In [ ]:
pd.crosstab(data['GroupSize'], data['CryoSleep'], normalize='index').style.format("{:,.0%}")

- People traveling alone have slightly lower than average chances to be Transported.
- There is a sweet spot for groups of 3 to 6 people with a Transporation rate around 60%.
- We see that groups have a higher percentage of cryo sleepers, which partly explains their higher transporation rate. Cryo Sleep is again the determining factor.  

To be thorough, we will now group people living in the same cabin. We do the same as we did for groups, but applied to cabin instead.  

In [ ]:
data['CabinNumber'] = data.Cabin.map(lambda p: str(p.split('/')[1]) + '_' + str(p.split('/')[0]))
data['CabinSize'] = data['CabinNumber'].map(data['CabinNumber'].value_counts())
data[data['CabinSize'] > 1].groupby('CabinNumber')['Transported'].mean().value_counts()

No pattern detected here. Less than half of the cabins in the ship had their occupants share the same fate.  


In [ ]:
data['Side'] = data.Cabin.map(lambda p: p.rsplit('/')[-1]) # Extracts the Side from Cabin column
data['Deck'] = data.Cabin.map(lambda p: p[:1]) # Extracts the Deck from Cabin column

pd.crosstab([data['Deck'], data['Side']], data['Transported'], normalize='index').style.format("{:,.0%}")

Now we check the decks and sides of the ship.
- Starboard side (right), has consistent higher transportation rates. It's clear the ship was hit harder on this side. The difference isn't huge, but worth keeping track of.
- Deck B is particularly dangerous. Deck C as well but mostly on the right side.
- Deck E, T and G's left side are the safest parts of the ship.

In [ ]:
pd.crosstab([data['HomePlanet'], data['Side']], data['Deck']).style.format("{:,.0f}")

This data shows that passengers from different planets live on separated decks.
- Europans are mostly on decks A, B & C
- Martians are mostly on decks D, E & F
- Earthians are mostly on deck F, G and a few on E
- T is meaningless, only 5 people.

In [ ]:
data.groupby(['Deck', 'HomePlanet'])['Transported'].mean()
pd.crosstab([data['Deck'], data['HomePlanet']], data['Transported'], normalize = 'index').style.format("{:,.0%}")

This gives us the transporation rate for passengers of each home planet, on each of the decks.
- Europans even on deck G had higher than average transporation rates.
- Marsians on Deck F had high transporation rates, while Earthians didn't.

In [ ]:
pd.crosstab([data['Deck'], data['CryoSleep']], data['Transported'], normalize='index').style.format("{:,.0%}")
# Calculates the transporation rare for people in cryosleep on each deck

This is a very interesting table, because it gives us very good percentages for the transported rate depending on the deck and cryosleep.  
Except for Deck E and G, passengers in cryosleep were almost certain to get Transported. 

## 5. Prediction
---

This is enough EDA to make a prediction now. Without using any library, we will try to make a small decision tree by hand, assigning points and weights to each relevant signal.  

Main signals: CryoSleep, luxury spending, deck, home planet, side, age for young children

Difficulty is finding the right weights for each element  


In [ ]:
avg_food = data['FoodCourt'].mean()
avg_shop = data['ShoppingMall'].mean()
avg_spa = data['Spa'].mean()
avg_vr = data['VRDeck'].mean()
avg_room = data['RoomService'].mean()

def prediction(data):
    survival = 0
    if data['Side'] == 'S': survival -=1
    if data['CryoSleep'] == True: survival -= 1
    if data['HomePlanet'] == 'Europa': survival -= 1
    if data['Deck'] in ('B', 'C'): survival -= 1
    if data['Age'] < 10: survival -= 1
    if data['FoodCourt'] > (avg_food * 3): survival -=1
    if data['ShoppingMall'] > (avg_shop * 2): survival -=1
    if data['Spa'] > avg_spa: survival +=3
    if data['VRDeck'] > avg_vr: survival +=3
    if data['RoomService'] > avg_room: survival +=2
    if survival < 0:
        return True
    else:
        return False
    
data['Prediction'] = data.apply(prediction, axis=1)

print((data['Prediction'] == data['Transported']).mean())

Creating the submission csv file:

In [ ]:
testdata = pd.read_csv('test.csv')  
fill_missing(testdata)
testdata['Side'] = testdata.Cabin.map(lambda p: p.rsplit('/')[-1])
testdata['Deck'] = testdata.Cabin.map(lambda p: p[:1])

testdata['Transported'] = testdata.apply(prediction, axis=1)

pd.DataFrame({
    'PassengerId': testdata['PassengerId'],
    'Transported': testdata['Transported']
}).to_csv('submission.csv', index=False)

## 6. Conclusion & Next Steps

---

Key findings:
- CryoSleep is the dominant predictor
- Luxury spending (Spa, VRDeck, RoomService) strongly predicts non-transport
- FoodCourt spending goes in the opposite direction
- Age < 10 is its own independant signal
- Deck, Home Planet, and Ship side increase probabilities to be transported
- Group and cabin analysis showed no strong pattern beyond CryoSleep

Manual prediction achieves 78.7% score.

Next: learn scikit-learn to improve prediction
